Cell 1, inisialisasi notebook

In [1]:
from pathlib import Path
import shutil
import polars as pl

# Lokasi dataset asli
DATA_DIR = Path(
    r"C:\Users\raihan bhakti\id_reg_kg_2511"
)

# Lokasi hasil cleaning, terpisah dari data asli
OUTPUT_DIR = DATA_DIR.parent / "id_reg_kg_2511_cleaned"

# Membaca daftar file sumber
parquet_files = sorted(DATA_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"Tidak ditemukan file Parquet di {DATA_DIR}"
    )

# Membaca dataset menggunakan LazyFrame
lf = pl.scan_parquet(
    [str(f) for f in parquet_files]
)

free_gb = shutil.disk_usage(DATA_DIR.parent).free / (1024 ** 3)

print("Versi Polars:", pl.__version__)
print("Jumlah file sumber:", len(parquet_files))
print("Folder sumber:", DATA_DIR)
print("Folder output:", OUTPUT_DIR)
print(f"Ruang disk kosong: {free_gb:.2f} GB")

Versi Polars: 1.44.2
Jumlah file sumber: 75
Folder sumber: C:\Users\raihan bhakti\id_reg_kg_2511
Folder output: C:\Users\raihan bhakti\id_reg_kg_2511_cleaned
Ruang disk kosong: 64.64 GB


Cell 2, audit kondisi sebelum cleaning

In [2]:
audit_sebelum = lf.select([
    pl.len().alias("total_baris"),

    (pl.col("year").str.strip_chars() == "0000")
    .sum()
    .alias("tahun_0000"),

    (pl.col("year").str.strip_chars() == "2030")
    .sum()
    .alias("tahun_2030"),

    (
        pl.col("regulation_type")
        .str.strip_chars()
        .str.to_lowercase() == "unknown"
    )
    .sum()
    .alias("regulation_type_unknown"),

    (
        pl.col("enacting_body")
        .str.strip_chars()
        .str.to_lowercase() == "unknown"
    )
    .sum()
    .alias("enacting_body_unknown"),

    pl.col("global_id")
    .is_null()
    .sum()
    .alias("global_id_null"),

    pl.col("global_id")
    .is_duplicated()
    .sum()
    .alias("global_id_berulang"),
]).collect(engine="streaming")

display(audit_sebelum)

total_baris,tahun_0000,tahun_2030,regulation_type_unknown,enacting_body_unknown,global_id_null,global_id_berulang
u32,u32,u32,u32,u32,u32,u32
748558,567,2,79764,64521,0,0


Cell 3, membuat fungsi cleaning

In [3]:
def build_clean_lazyframe(source_lf):

    exprs = []

    # 1. Membersihkan spasi dan string kosong
    text_cols = [
        "regulation_number",
        "effective_date",
        "chapter",
        "article",
        "about",
        "content",
    ]

    for col in text_cols:
        value = pl.col(col).str.strip_chars()

        exprs.append(
            pl.when(value == "")
            .then(pl.lit(None, dtype=pl.String))
            .otherwise(value)
            .alias(col)
        )

    # 2. Menangani kategori Unknown
    category_cols = [
        "regulation_type",
        "enacting_body",
    ]

    for col in category_cols:
        value = pl.col(col).str.strip_chars()

        is_unknown = (
            value.str.to_lowercase() == "unknown"
        )

        exprs.append(
            pl.when(
                value.is_null()
                | (value == "")
                | is_unknown
            )
            .then(pl.lit(None, dtype=pl.String))
            .otherwise(value)
            .alias(col)
        )

    # 3. Memeriksa tahun sebelum perubahan
    raw_year = (
        pl.col("year")
        .str.strip_chars()
        .cast(pl.Int32, strict=False)
    )

    # Tahun 0000 dijadikan null
    exprs.append(
        pl.when(raw_year == 0)
        .then(pl.lit(None, dtype=pl.Int32))
        .otherwise(raw_year)
        .alias("year")
    )

    # 4. Menyimpan status pemeriksaan tahun
    exprs.append(
        pl.when(raw_year.is_null())
        .then(pl.lit("missing_or_unparseable"))

        .when(raw_year == 0)
        .then(pl.lit("invalid_sentinel_0000"))

        .when(raw_year > 2026)
        .then(pl.lit("future_year_check"))

        .when(raw_year < 1800)
        .then(pl.lit("historical_year_check"))

        .otherwise(pl.lit("ok"))
        .alias("year_status")
    )

    return source_lf.with_columns(exprs)


print("Fungsi cleaning berhasil disiapkan.")

Fungsi cleaning berhasil disiapkan.


Cell 4, menjalankan cleaning secara virtual dan memeriksa hasilnya

In [4]:
clean_lf = build_clean_lazyframe(lf)

audit_sesudah = clean_lf.select([
    pl.len().alias("total_baris"),

    pl.col("year")
    .is_null()
    .sum()
    .alias("year_null_setelah"),

    pl.col("regulation_type")
    .is_null()
    .sum()
    .alias("regulation_type_null_setelah"),

    pl.col("enacting_body")
    .is_null()
    .sum()
    .alias("enacting_body_null_setelah"),

    pl.col("global_id")
    .is_null()
    .sum()
    .alias("global_id_null_setelah"),

    pl.col("global_id")
    .is_duplicated()
    .sum()
    .alias("global_id_berulang_setelah"),
]).collect(engine="streaming")

display(audit_sesudah)

total_baris,year_null_setelah,regulation_type_null_setelah,enacting_body_null_setelah,global_id_null_setelah,global_id_berulang_setelah
u32,u32,u32,u32,u32,u32
748558,567,79764,64521,0,0


Cell 5, memeriksa status tahun hasil cleaning

In [5]:
year_status_counts = (
    clean_lf
    .group_by("year_status")
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort(
        "jumlah_baris",
        descending=True
    )
    .collect(engine="streaming")
)

display(year_status_counts)

year_status,jumlah_baris
str,u32
"""ok""",747989
"""invalid_sentinel_0000""",567
"""future_year_check""",2


Cell 6, memeriksa contoh hasil cleaning

In [6]:
contoh_cleaning = (
    clean_lf
    .filter(
        (pl.col("year_status") != "ok")
        | pl.col("regulation_type").is_null()
        | pl.col("enacting_body").is_null()
    )
    .select([
        "global_id",
        "regulation_type",
        "enacting_body",
        "year",
        "year_status",
        "effective_date",
        "about",
    ])
    .head(10)
    .collect(engine="streaming")
)

display(contoh_cleaning)

global_id,regulation_type,enacting_body,year,year_status,effective_date,about
i64,str,str,i32,str,str,str
47,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
48,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
49,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
50,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
51,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
52,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
53,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
54,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"
55,null,"""REPUBLIK INDONESIA""",2006,"""ok""","""15 Juni 2016""","""TATAKERJA BADANPEMERIKSA KEUAN…"


In [7]:
# Cell 7: Ekspor hasil cleaning
# Jalankan setelah audit hasil cleaning diperiksa.

free_gb = shutil.disk_usage(DATA_DIR.parent).free / (1024 ** 3)

print(f"Ruang disk kosong: {free_gb:.2f} GB")

if free_gb < 10:
    print(
        "EKSPOR DITUNDA. Ruang disk kosong kurang dari "
        "10 GB. Jangan menulis salinan dataset penuh dahulu."
    )

else:
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    for i, src in enumerate(parquet_files, start=1):

        dst = OUTPUT_DIR / src.name
        temp = OUTPUT_DIR / f"{src.stem}.tmp.parquet"

        if temp.exists():
            temp.unlink()

        build_clean_lazyframe(
            pl.scan_parquet(str(src))
        ).sink_parquet(
            str(temp),
            compression="zstd"
        )

        if dst.exists():
            dst.unlink()

        temp.replace(dst)

        print(
            f"[{i}/{len(parquet_files)}] "
            f"Selesai: {src.name}"
        )

    print("Ekspor selesai.")
    print("Lokasi hasil:", OUTPUT_DIR)

Ruang disk kosong: 64.63 GB
[1/75] Selesai: train-00001-of-99999.parquet
[2/75] Selesai: train-00002-of-99999.parquet
[3/75] Selesai: train-00003-of-99999.parquet
[4/75] Selesai: train-00004-of-99999.parquet
[5/75] Selesai: train-00005-of-99999.parquet
[6/75] Selesai: train-00006-of-99999.parquet
[7/75] Selesai: train-00007-of-99999.parquet
[8/75] Selesai: train-00008-of-99999.parquet
[9/75] Selesai: train-00009-of-99999.parquet
[10/75] Selesai: train-00010-of-99999.parquet
[11/75] Selesai: train-00011-of-99999.parquet
[12/75] Selesai: train-00012-of-99999.parquet
[13/75] Selesai: train-00013-of-99999.parquet
[14/75] Selesai: train-00014-of-99999.parquet
[15/75] Selesai: train-00015-of-99999.parquet
[16/75] Selesai: train-00016-of-99999.parquet
[17/75] Selesai: train-00017-of-99999.parquet
[18/75] Selesai: train-00018-of-99999.parquet
[19/75] Selesai: train-00019-of-99999.parquet
[20/75] Selesai: train-00020-of-99999.parquet
[21/75] Selesai: train-00021-of-99999.parquet
[22/75] Selesai

Cell 8, validasi hasil yang tersimpan

In [10]:
# Cell 8: Validasi hasil cleaning yang tersimpan

cleaned_files = sorted(
    OUTPUT_DIR.glob("*.parquet")
)

if len(cleaned_files) != len(parquet_files):
    raise ValueError(
        f"Jumlah file tidak sesuai. "
        f"Sumber: {len(parquet_files)}, "
        f"hasil: {len(cleaned_files)}"
    )

cleaned_lf = pl.scan_parquet(
    [str(f) for f in cleaned_files]
)

validasi_output = cleaned_lf.select([
    pl.len().alias("total_baris"),

    pl.col("global_id")
    .n_unique()
    .alias("global_id_unik"),

    pl.col("global_id")
    .is_null()
    .sum()
    .alias("global_id_null"),

    pl.col("global_id")
    .is_duplicated()
    .sum()
    .alias("global_id_berulang"),

    pl.col("year")
    .is_null()
    .sum()
    .alias("year_null"),

    pl.col("year_status")
    .is_null()
    .sum()
    .alias("year_status_null"),
]).collect(engine="streaming")

display(validasi_output)

print("Jumlah file hasil:", len(cleaned_files))
print("Lokasi hasil:", OUTPUT_DIR)

total_baris,global_id_unik,global_id_null,global_id_berulang,year_null,year_status_null
u32,u32,u32,u32,u32,u32
748558,748558,0,0,567,0


Jumlah file hasil: 75
Lokasi hasil: C:\Users\raihan bhakti\id_reg_kg_2511_cleaned


Cell 9 berikut.

In [11]:

# Cell 9: Validasi kategori dan status tahun

validasi_kategori = cleaned_lf.select([
    pl.len().alias("total_baris"),

    (
        pl.col("regulation_type")
        .str.to_lowercase() == "unknown"
    ).sum().alias("regulation_type_unknown_sisa"),

    (
        pl.col("enacting_body")
        .str.to_lowercase() == "unknown"
    ).sum().alias("enacting_body_unknown_sisa"),

    pl.col("regulation_type")
    .is_null()
    .sum()
    .alias("regulation_type_null"),

    pl.col("enacting_body")
    .is_null()
    .sum()
    .alias("enacting_body_null"),

    (pl.col("year") == 2030)
    .sum()
    .alias("tahun_2030_dipertahankan"),

    (pl.col("year_status") == "invalid_sentinel_0000")
    .sum()
    .alias("tahun_0000_ditandai"),

    (pl.col("year_status") == "future_year_check")
    .sum()
    .alias("tahun_masa_depan_ditandai"),
]).collect(engine="streaming")

display(validasi_kategori)


total_baris,regulation_type_unknown_sisa,enacting_body_unknown_sisa,regulation_type_null,enacting_body_null,tahun_2030_dipertahankan,tahun_0000_ditandai,tahun_masa_depan_ditandai
u32,u32,u32,u32,u32,u32,u32,u32
748558,0,0,79764,64521,2,567,2


Cell 10. Ringkasan kualitas dataset hasil cleaning

In [12]:

# Cell 10: Data Quality Summary menggunakan DuckDB

import duckdb

CLEAN_PATH = str(
    OUTPUT_DIR / "*.parquet"
).replace("\\", "/")

con = duckdb.connect()

query = f"""
SUMMARIZE
SELECT
    global_id,
    regulation_type,
    enacting_body,
    regulation_number,
    year,
    year_status,
    effective_date,
    length(about) AS about_chars,
    length(content) AS content_chars
FROM read_parquet(
    '{CLEAN_PATH}',
    union_by_name = true
)
"""

summary_duckdb = con.execute(query).df()

display(summary_duckdb)


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,global_id,BIGINT,0,748586,795829,374287.72937300784,216099.37190603316,186877,373605,562641,748558,0.00
1,regulation_type,VARCHAR,INSTRUKSI PRESIDEN,UUD,32,None,None,None,None,None,748558,10.66
2,enacting_body,VARCHAR,! BOALEMO,� I � R � T7EFI �� ?M � ARA TANGGAL 31 DESEMBE...,2681,None,None,None,None,None,748558,8.62
3,regulation_number,VARCHAR,-,ZS,1692,None,None,None,None,None,748558,0.00
4,year,INTEGER,1945,2030,76,2009.2470016350464,12.525976496234328,2009,2013,2015,748558,0.08
5,year_status,VARCHAR,future_year_check,ok,3,None,None,None,None,None,748558,0.00
6,effective_date,VARCHAR,01 Agustus 2001,Unknown,5085,None,None,None,None,None,748558,0.00
7,about_chars,BIGINT,1,32767,2290,201.25269918964196,814.0395384349375,36,67,116,748558,0.00
8,content_chars,BIGINT,3,32767,14537,1234.3257476374577,3330.776802319558,188,421,918,748558,0.00


Cell 11, audit kandidat masalah tambahan

In [13]:

# Cell 11: Audit kandidat masalah metadata

audit_metadata = cleaned_lf.select([
    pl.len().alias("total_baris"),

    (
        pl.col("effective_date")
        .str.to_lowercase() == "unknown"
    ).sum().alias("effective_date_unknown"),

    (
        pl.col("enacting_body")
        .str.contains("�", literal=True)
        .fill_null(False)
    ).sum().alias("enacting_body_karakter_pengganti"),

    (
        pl.col("regulation_number")
        .str.contains("�", literal=True)
        .fill_null(False)
    ).sum().alias("regulation_number_karakter_pengganti"),
]).collect(engine="streaming")

display(audit_metadata)

# Menampilkan maksimal 10 contoh lembaga penerbit
contoh_encoding = (
    cleaned_lf
    .filter(
        pl.col("enacting_body")
        .str.contains("�", literal=True)
        .fill_null(False)
    )
    .select([
        "global_id",
        "regulation_type",
        "enacting_body",
        "regulation_number",
        "year",
    ])
    .head(10)
    .collect(engine="streaming")
)

display(contoh_encoding)


total_baris,effective_date_unknown,enacting_body_karakter_pengganti,regulation_number_karakter_pengganti
u32,u32,u32,u32
748558,259467,61,0


global_id,regulation_type,enacting_body,regulation_number,year
i64,str,str,str,i32
40301,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40302,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40303,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40304,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40305,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40306,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
40307,"""PERATURAN BUPATI""","""SINTANG TENT ANG PETU � JUK TE…","""63""",2013
85932,"""PERATURAN BUPATI""","""BUTON NUMU � G6 TAHUN 2016 TEN…","""43""",2016
85933,"""PERATURAN BUPATI""","""BUTON NUMU � G6 TAHUN 2016 TEN…","""43""",2016


Cell 12. Finalisasi cleaning, validasi, dan laporan otomatis

In [14]:
# CELL 12: FINAL CLEANING, VALIDATION, AND REPORT

from pathlib import Path
from datetime import datetime
import json
import polars as pl

# Folder hasil cleaning, bukan folder sumber asli
cleaned_files = sorted(OUTPUT_DIR.glob("*.parquet"))

assert len(cleaned_files) == 75, (
    f"Jumlah file tidak sesuai: {len(cleaned_files)}"
)

print("Memperbarui metadata tanggal dan encoding...")
print("File yang diproses:", len(cleaned_files))

# Perbarui setiap file secara bertahap
for i, file_path in enumerate(cleaned_files, start=1):

    temp_path = file_path.with_name(
        file_path.stem + ".tmp"
    )

    if temp_path.exists():
        temp_path.unlink()

    source = pl.scan_parquet(str(file_path))

    date_value = pl.col("effective_date").str.strip_chars()

    date_unknown = (
        date_value.is_null()
        | (date_value == "")
        | (date_value.str.to_lowercase() == "unknown")
    )

    updated = source.with_columns([
        # Unknown pada tanggal menjadi null
        pl.when(date_unknown)
        .then(pl.lit(None, dtype=pl.String))
        .otherwise(date_value)
        .alias("effective_date"),

        # Status tanggal tetap terdokumentasi
        pl.when(date_unknown)
        .then(pl.lit("unknown_or_missing"))
        .otherwise(pl.lit("present"))
        .alias("effective_date_status"),

        # Menandai karakter pengganti tanpa mengubah teks asli
        pl.col("enacting_body")
        .str.contains("�", literal=True)
        .fill_null(False)
        .alias("enacting_body_encoding_issue"),
    ])

    # Tulis ke file sementara terlebih dahulu
    updated.sink_parquet(
        str(temp_path),
        compression="zstd"
    )

    # Ganti file lama hanya setelah file sementara selesai
    temp_path.replace(file_path)

    print(f"[{i}/75] Selesai: {file_path.name}")

print("\nPembaruan seluruh file selesai.")

# Baca ulang seluruh hasil yang sudah diperbarui
cleaned_files = sorted(OUTPUT_DIR.glob("*.parquet"))

cleaned_lf = pl.scan_parquet(
    [str(f) for f in cleaned_files]
)

# Validasi akhir seluruh dataset
final_audit = cleaned_lf.select([
    pl.len().alias("total_baris"),

    pl.col("global_id")
    .n_unique()
    .alias("global_id_unik"),

    pl.col("global_id")
    .is_null()
    .sum()
    .alias("global_id_null"),

    pl.col("global_id")
    .is_duplicated()
    .sum()
    .alias("global_id_berulang"),

    pl.col("year")
    .is_null()
    .sum()
    .alias("year_null"),

    pl.col("year_status")
    .is_null()
    .sum()
    .alias("year_status_null"),

    (
        pl.col("regulation_type")
        .str.to_lowercase() == "unknown"
    ).fill_null(False).sum()
    .alias("regulation_type_unknown_sisa"),

    (
        pl.col("enacting_body")
        .str.to_lowercase() == "unknown"
    ).fill_null(False).sum()
    .alias("enacting_body_unknown_sisa"),

    (
        pl.col("effective_date")
        .str.to_lowercase() == "unknown"
    ).fill_null(False).sum()
    .alias("effective_date_unknown_sisa"),

    pl.col("effective_date")
    .is_null()
    .sum()
    .alias("effective_date_null"),

    (
        pl.col("effective_date_status")
        == "unknown_or_missing"
    ).sum()
    .alias("effective_date_unknown_ditandai"),

    pl.col("enacting_body_encoding_issue")
    .sum()
    .alias("enacting_body_encoding_flag"),

    (pl.col("year") == 2030)
    .sum()
    .alias("tahun_2030_dipertahankan"),

    (
        pl.col("year_status")
        == "invalid_sentinel_0000"
    ).sum()
    .alias("tahun_0000_ditandai"),

    (
        pl.col("year_status")
        == "future_year_check"
    ).sum()
    .alias("tahun_masa_depan_ditandai"),
]).collect(engine="streaming")

display(final_audit)

# Pemeriksaan otomatis
result = final_audit.to_dicts()[0]

assert result["total_baris"] == 748558
assert result["global_id_unik"] == 748558
assert result["global_id_null"] == 0
assert result["global_id_berulang"] == 0
assert result["year_null"] == 567
assert result["year_status_null"] == 0
assert result["regulation_type_unknown_sisa"] == 0
assert result["enacting_body_unknown_sisa"] == 0
assert result["effective_date_unknown_sisa"] == 0
assert result["effective_date_unknown_ditandai"] == 259467
assert result["enacting_body_encoding_flag"] == 61
assert result["tahun_2030_dipertahankan"] == 2
assert result["tahun_0000_ditandai"] == 567
assert result["tahun_masa_depan_ditandai"] == 2

# Buat laporan kualitas data
total_bytes = sum(f.stat().st_size for f in cleaned_files)

report = {
    "project": "Indonesian Regulation Dataset",
    "stage": "Data Cleaning Validation",
    "generated_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "output_directory": str(OUTPUT_DIR),
    "file_count": len(cleaned_files),
    "size_gib": round(total_bytes / (1024 ** 3), 2),
    "validation": result,
    "cleaning_notes": [
        "Original source files remain unchanged.",
        "Year 0000 is represented as null and flagged.",
        "Year 2030 is retained and flagged for review.",
        "Unknown category labels are represented as null.",
        "Unknown effective dates are represented as null and flagged.",
        "Potential encoding issues are flagged without modifying source text.",
        "No rows were intentionally deleted."
    ]
}

report_path = OUTPUT_DIR / "data_quality_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n=== FINAL VALIDATION ===")
print("Jumlah file:", len(cleaned_files))
print("Total baris:", result["total_baris"])
print("Ukuran hasil:", report["size_gib"], "GiB")
print("Laporan:", report_path)
print("STATUS: SELURUH ASSERTION BERHASIL")

Memperbarui metadata tanggal dan encoding...
File yang diproses: 75
[1/75] Selesai: train-00001-of-99999.parquet
[2/75] Selesai: train-00002-of-99999.parquet
[3/75] Selesai: train-00003-of-99999.parquet
[4/75] Selesai: train-00004-of-99999.parquet
[5/75] Selesai: train-00005-of-99999.parquet
[6/75] Selesai: train-00006-of-99999.parquet
[7/75] Selesai: train-00007-of-99999.parquet
[8/75] Selesai: train-00008-of-99999.parquet
[9/75] Selesai: train-00009-of-99999.parquet
[10/75] Selesai: train-00010-of-99999.parquet
[11/75] Selesai: train-00011-of-99999.parquet
[12/75] Selesai: train-00012-of-99999.parquet
[13/75] Selesai: train-00013-of-99999.parquet
[14/75] Selesai: train-00014-of-99999.parquet
[15/75] Selesai: train-00015-of-99999.parquet
[16/75] Selesai: train-00016-of-99999.parquet
[17/75] Selesai: train-00017-of-99999.parquet
[18/75] Selesai: train-00018-of-99999.parquet
[19/75] Selesai: train-00019-of-99999.parquet
[20/75] Selesai: train-00020-of-99999.parquet
[21/75] Selesai: trai

total_baris,global_id_unik,global_id_null,global_id_berulang,year_null,year_status_null,regulation_type_unknown_sisa,enacting_body_unknown_sisa,effective_date_unknown_sisa,effective_date_null,effective_date_unknown_ditandai,enacting_body_encoding_flag,tahun_2030_dipertahankan,tahun_0000_ditandai,tahun_masa_depan_ditandai
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
748558,748558,0,0,567,0,0,0,0,259467,259467,61,2,567,2



=== FINAL VALIDATION ===
Jumlah file: 75
Total baris: 748558
Ukuran hasil: 2.92 GiB
Laporan: C:\Users\raihan bhakti\id_reg_kg_2511_cleaned\data_quality_report.json
STATUS: SELURUH ASSERTION BERHASIL
